In [17]:
import os
import zipfile
import shutil
from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import cv2  # Added for blur detection
from IPython.display import display, FileLink

#   - ZIP files
#   - Folders containing MRI images
#   - Individual image files

SOURCE_PATHS = [

    Path(r"C:\Users\USER\Downloads\CodeDataS\Mri_clean_Stroke_224x224.zip"),

    # Add more files/folders here if needed
    # Path(r"C:\Users\USER\Downloads\MRI_Dataset_3.zip"),
    # Path(r"C:\Users\USER\Downloads\Extra_MRI_Images"),

]


# Temporary folder containing cleaned images
CLEAN_DIR = Path("Mri_clean_images")

# Name of final ZIP file
ZIP_BASE_NAME = "Mri_clean_224x224"

# Final image size
TARGET_SIZE = (224, 224)

# Black padding
PADDING_COLOR = 0

# Supported image formats
IMAGE_EXTENSIONS = [
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
]


BLUR_THRESHOLD = 50


print("=" * 65)
print("🧠 MRI DATASET CLEANING AND PREPROCESSING")
print("=" * 65)
print()

if CLEAN_DIR.exists():

    try:
        shutil.rmtree(CLEAN_DIR)

    except PermissionError:

        print(
            "❌ Permission denied while removing "
            "the previous clean-image folder."
        )

        print(
            "Please close the folder in File Explorer "
            "and run the cell again."
        )

        raise


CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ZIP_OUTPUT = Path(
    f"{ZIP_BASE_NAME}.zip"
)

counter = 1

while ZIP_OUTPUT.exists():

    ZIP_OUTPUT = Path(
        f"{ZIP_BASE_NAME}_{counter}.zip"
    )

    counter += 1


print(
    f"📦 Final ZIP will be created as: "
    f"{ZIP_OUTPUT}"
)

print()


hash_registry = {}

metrics = {
    "valid": 0,
    "corrupted": 0,
    "duplicate": 0,
    "blurry": 0,
    "already_grayscale": 0,
    "converted_to_grayscale": 0
}


def calculate_dhash(image, hash_size=8):

    """
    Generates a visual gradient hash.

    This is used to identify duplicate or
    visually similar MRI images.
    """

    img = image.convert("L").resize(
        (hash_size + 1, hash_size),
        Image.Resampling.BILINEAR
    )

    pixels = np.array(
        img,
        dtype=float
    )

    diff = (
        pixels[:, 1:]
        >
        pixels[:, :-1]
    )

    return ''.join(
        f'{int(b)}'
        for b in diff.flatten()
    )


def resize_with_padding(
    img,
    target_size,
    pad_color
):

    """
    Resizes an MRI image while maintaining
    its original aspect ratio.

    Padding is added so that the final image
    is exactly 512 x 512 pixels.
    """

    
    img = img.copy()

    
    img.thumbnail(
        target_size,
        Image.Resampling.LANCZOS
    )

   
    delta_w = (
        target_size[0]
        -
        img.size[0]
    )

    delta_h = (
        target_size[1]
        -
        img.size[1]
    )

    left = delta_w // 2
    right = delta_w - left

    top = delta_h // 2
    bottom = delta_h - top

    padding = (
        left,
        top,
        right,
        bottom
    )

    
    processed_img = ImageOps.expand(
        img,
        padding,
        fill=pad_color
    )

    
    processed_img = processed_img.resize(
        target_size,
        Image.Resampling.LANCZOS
    )

    return processed_img


def process_image_stream(
    file_stream,
    original_name,
    parent_context=""
):

    """
    Validates, checks duplicates, resizes,
    and saves an individual MRI image.
    """

    try:

        with Image.open(file_stream) as img:

            img.verify()


        file_stream.seek(0)


        with Image.open(file_stream) as img:


            # Check whether the image is already grayscale
            rgb_image = img.convert("RGB")
            rgb_array = np.array(rgb_image)

            # A grayscale image has equal red, green and blue
            # values for every pixel.
            is_grayscale = np.array_equal(
                rgb_array[:, :, 0],
                rgb_array[:, :, 1]
            ) and np.array_equal(
                rgb_array[:, :, 1],
                rgb_array[:, :, 2]
            )

            if is_grayscale:
                metrics["already_grayscale"] += 1
                print(f"⚫ Already grayscale: {original_name}")
            else:
                metrics["converted_to_grayscale"] += 1
                print(f"🎨 Converting to grayscale: {original_name}")

            # All images will now be single-channel grayscale
            img = rgb_image.convert("L")

            grayscale_image = np.array(img)

            blur_score = cv2.Laplacian(
                grayscale_image, cv2.CV_64F
            ).var()

            if blur_score < BLUR_THRESHOLD:
                metrics["blurry"] += 1
                print(
                    f"🌫️ Blurry image skipped: {original_name} "
                    f"(score: {blur_score:.2f})"
                )
                return

            img_hash = calculate_dhash(img)


            if img_hash in hash_registry:

                metrics["duplicate"] += 1

                return


            processed_img = resize_with_padding(
                img,
                TARGET_SIZE,
                PADDING_COLOR
            )

            
            original_file_name = Path(
                original_name
            ).stem

            original_file_name = (
                original_file_name
                .replace(" ", "_")
                .replace("/", "_")
                .replace("\\", "_")
                .replace(":", "_")
            )

            parent_context = (
                parent_context
                .replace(" ", "_")
                .replace("/", "_")
                .replace("\\", "_")
                .replace(":", "_")
            )

            clean_name = (
                f"clean_{parent_context}_"
                f"{original_file_name}.jpg"
            )


            new_path = (
                CLEAN_DIR
                /
                clean_name
            )

            filename_counter = 1

            while new_path.exists():

                clean_name = (
                    f"clean_{parent_context}_"
                    f"{original_file_name}_"
                    f"{filename_counter}.jpg"
                )

                new_path = (
                    CLEAN_DIR
                    /
                    clean_name
                )

                filename_counter += 1


            processed_img.save(
                new_path,
                "JPEG",
                quality=95
            )


            hash_registry[img_hash] = new_path

            metrics["valid"] += 1


    except Exception as e:

        print(
            f"❌ Error processing: "
            f"{original_name} "
            f"in {parent_context} -> {e}"
        )

        metrics["corrupted"] += 1


def process_zip_file(zip_path):

    """
    Processes all supported MRI images inside a ZIP file.
    """

    print(
        f"📦 Processing ZIP: "
        f"{zip_path.name}"
    )

    try:

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as archive:

            for file_info in archive.infolist():

                
                if file_info.is_dir():
                    continue

                
                file_path = Path(
                    file_info.filename
                )

                
                if file_path.name.startswith("."):
                    continue

                
                if (
                    file_path.suffix.lower()
                    not in IMAGE_EXTENSIONS
                ):
                    continue


               
                with archive.open(
                    file_info
                ) as file_stream:

                    process_image_stream(
                        file_stream,
                        file_info.filename,
                        parent_context=zip_path.stem
                    )


    except zipfile.BadZipFile:

        print(
            f"❌ Invalid ZIP file: "
            f"{zip_path}"
        )

        metrics["corrupted"] += 1




def process_image_file(image_path):

    """
    Processes an individual MRI image.
    """

    print(
        f"🖼️ Processing image: "
        f"{image_path.name}"
    )

    try:

        with open(
            image_path,
            "rb"
        ) as file_stream:

            process_image_stream(
                file_stream,
                image_path.name,
                parent_context=image_path.parent.name
            )

    except Exception as e:

        print(
            f"❌ Could not open "
            f"{image_path}: {e}"
        )

        metrics["corrupted"] += 1


def process_folder(folder_path):

    """
    Recursively searches a folder and processes
    all supported images and ZIP files.
    """

    print(
        f"📁 Processing folder: "
        f"{folder_path}"
    )

    files_found = list(
        folder_path.rglob("*")
    )

    for target in files_found:

        
        if target.is_dir():
            continue

        
        if target.name.startswith("."):
            continue


        
        if target.suffix.lower() == ".zip":

            process_zip_file(target)


        
        elif (
            target.suffix.lower()
            in IMAGE_EXTENSIONS
        ):

            process_image_file(target)


print("🔎 Checking input files and folders...")
print()

if len(SOURCE_PATHS) == 0:

    raise ValueError(
        "❌ No input files or folders were added."
    )


valid_inputs = 0


for source in SOURCE_PATHS:

    if source.exists():

        valid_inputs += 1

        print(
            f"✅ Found: {source}"
        )

    else:

        print(
            f"⚠️ NOT FOUND: {source}"
        )


print()

if valid_inputs == 0:

    raise FileNotFoundError(
        "❌ None of the specified files/folders "
        "could be found. Please check your paths."
    )


print("=" * 65)
print("🚀 STARTING DATASET PROCESSING")
print("=" * 65)
print()


for source in SOURCE_PATHS:

    if not source.exists():

        continue


    if source.is_file() and source.suffix.lower() == ".zip":

        process_zip_file(source)


    elif (
        source.is_file()
        and
        source.suffix.lower()
        in IMAGE_EXTENSIONS
    ):

        process_image_file(source)


    elif source.is_dir():

        process_folder(source)


    else:

        print(
            f"⚠️ Unsupported input: "
            f"{source}"
        )


print()
print("=" * 65)
print("🗜️ CREATING FINAL CLEAN ZIP")
print("=" * 65)
print()

if metrics["valid"] == 0:

    print(
        "❌ No valid images were processed."
    )

    print(
        "Please check your dataset paths and "
        "image formats."
    )

else:

   
    shutil.make_archive(
        base_name=ZIP_OUTPUT.with_suffix(""),
        format="zip",
        root_dir=CLEAN_DIR
    )


print()
print("=" * 65)
print("📊 MRI DATASET CLEANING COMPLETE")
print("=" * 65)

print(
    f"✅ Unique images processed: "
    f"{metrics['valid']}"
)

print(
    f"👥 Duplicate images removed: "
    f"{metrics['duplicate']}"
)


print(
    f"🌫️ Blurry images skipped: "
    f"{metrics['blurry']}"
)

print(
    f"⚫ Images already grayscale: "
    f"{metrics['already_grayscale']}"
)

print(
    f"🎨 Images converted to grayscale: "
    f"{metrics['converted_to_grayscale']}"
)

print(
    f"❌ Corrupted/unreadable images: "
    f"{metrics['corrupted']}"
)

print(
    f"📐 Final image dimensions: "
    f"{TARGET_SIZE[0]} x {TARGET_SIZE[1]}"
)


if ZIP_OUTPUT.exists():

    zip_size_mb = (
        ZIP_OUTPUT.stat().st_size
        /
        (1024 * 1024)
    )

    print(
        f"🗜️ ZIP file: "
        f"{ZIP_OUTPUT}"
    )

    print(
        f"📦 ZIP size: "
        f"{zip_size_mb:.2f} MB"
    )

    print()
    print("=" * 65)
    print("⬇️ DOWNLOAD YOUR CLEAN DATASET")
    print("=" * 65)
    print()

    display(
        FileLink(
            str(ZIP_OUTPUT),
            result_html_prefix="👉 "
        )
    )

else:

    print()
    print(
        "❌ The ZIP file could not be created."
    )


print()
print(
    f"📁 Temporary cleaned images are stored in: "
    f"{CLEAN_DIR}"
)

print()
print("🎉 Processing finished successfully!")

🧠 MRI DATASET CLEANING AND PREPROCESSING

📦 Final ZIP will be created as: Mri_clean_224x224_1.zip

🔎 Checking input files and folders...

✅ Found: C:\Users\USER\Downloads\CodeDataS\Mri_clean_Stroke_224x224.zip

🚀 STARTING DATASET PROCESSING

📦 Processing ZIP: Mri_clean_Stroke_224x224.zip
🎨 Converting to grayscale: clean_Final_stroke_dataset_10017_2_Ischemic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10052_4_Hemorrhagic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10060_9_Ischemic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10062_7_Ischemic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10077_9_Ischemic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10101_3_Ischemic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10134_4_Hemorrhagic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_dataset_10143_1_Hemorrhagic_Stroke.jpg
🎨 Converting to grayscale: clean_Final_stroke_

C:\Users\USER\Downloads\CodeDataS\Mri_clean_224x224_1.zip


📁 Temporary cleaned images are stored in: Mri_clean_images

🎉 Processing finished successfully!
